___
___
# FASE 3: Analisis Mensual Agrupado
___
___


___
## 1. Librerias
___

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

___
## 2. Configuración visual
___

In [19]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

___
## 3. Data
___

### 3.1. Importar data.

In [20]:
df_ventas = pd.read_csv('data/ventas_diarias.csv')
df_compras = pd.read_csv('data/compras_proveedores.csv')

### 3.2. Convertir fechas a formato datatime para poder operar cronologicamente

In [21]:
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_compras['fecha'] = pd.to_datetime(df_compras['fecha'])

### 3.3. Calcular el costo de reposicion real de cada producto

In [22]:
df_compras = df_compras.sort_values('fecha')
def obtener_costo_reposicion(row):
    compras_previas = df_compras[(df_compras['producto_id'] == row['producto_id']) & (df_compras['fecha'] <= row['fecha'])]
    return compras_previas.iloc[-1]['costo_unitario'] if not compras_previas.empty else row['costo_asociado_simulado']

df_ventas['costo_reposicion'] = df_ventas.apply(obtener_costo_reposicion, axis=1)
df_ventas['margen_unitario_usd'] = df_ventas['precio_unitario'] - df_ventas['costo_reposicion']
df_ventas['fuga_transaccion_usd'] = np.where(df_ventas['margen_unitario_usd'] < 0, abs(df_ventas['margen_unitario_usd']) * df_ventas['cantidad_vendida'], 0)


### 3.4. Extraer el período (Año-Mes) para el agrupamiento

In [23]:
df_ventas['periodo'] = df_ventas['fecha'].dt.to_period('M')

___
## 4. Agrupamiento Macro Mensual por Producto
___

### 4.1. Tabulacion del Agrupamiento.

In [24]:
df_agrupado = df_ventas.groupby(['periodo', 'producto_id']).agg(
    unidades_vendidas=('cantidad_vendida', 'sum'),
    ingreso_total_usd=('precio_unitario', lambda x: (x * df_ventas.loc[x.index, 'cantidad_vendida']).sum()),
    fuga_total_usd=('fuga_transaccion_usd', 'sum'),
    margen_promedio_percent=('margen_unitario_usd', lambda x: (x / df_ventas.loc[x.index, 'precio_unitario']).mean() * 100)
).reset_index()

### 4.2. Convertir período a string para facilitar la graficación.

In [25]:
df_agrupado['periodo'] = df_agrupado['periodo'].astype(str)

___
## 5. Visualizacion de Resultados
___


### 5.1. Matriz de Calor (Heatmap) de Degradación de Márgenes.

In [26]:
pivot_margen = df_agrupado.pivot(index='producto_id', columns='periodo', values='margen_promedio_percent')

plt.figure(figsize=(10, 5))
sns.heatmap(pivot_margen, annot=True, annot_kws={"size":8}, fmt=".1f", cmap="RdYlGn", yticklabels=True, cbar_kws={'label': 'Margen Bruto Promedio (%)'})
plt.title('Mapa de Calor: Evolución Mensual del Margen Bruto por Producto\n(Zonas rojas indican pérdida crítica de rentabilidad)', fontsize=12, fontweight='bold', pad=15)
plt.ylabel('Código de Producto', fontsize=10)
plt.xlabel('Mes de Operación')
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig('figures/heatmap_degradacion_margen.png', dpi=300)
plt.close()

### 5.2. Tendencia Acumulada de la Pérdida.

In [27]:
fuga_mensual_total = df_agrupado.groupby('periodo')['fuga_total_usd'].sum().reset_index()

plt.figure(figsize=(10, 4))
ax = sns.lineplot(x='periodo', y='fuga_total_usd', data=fuga_mensual_total, marker='o', color='#D32F2F', linewidth=3)
plt.fill_between(fuga_mensual_total['periodo'], fuga_mensual_total['fuga_total_usd'], color='#D32F2F', alpha=0.1)
plt.title('Tendencia de la Fuga Financiera Mensual Acumulada\n(El costo real del descontrol interno)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Mes de Operación')
plt.ylabel('Pérdida Total (USD)')

for x, y in zip(fuga_mensual_total['periodo'], fuga_mensual_total['fuga_total_usd']):
    plt.text(x, y + (fuga_mensual_total['fuga_total_usd'].max()*0.03), f'${y:,.2f}', ha='center', fontweight='bold', color='#D32F2F')

plt.tight_layout()
plt.savefig('figures/tendencia_fuga_mensual.png', dpi=300)
plt.close()

print("¡Notebook 3 completado! Gráficas de tendencia y mapas de calor exportados.")

¡Notebook 3 completado! Gráficas de tendencia y mapas de calor exportados.
